# 06 · 스트림과 비동기 처리

> **CuPy 2일 집중 코스 — Day 1 / 단원 4 (스트림과 비동기 처리) — Day 1 마무리**

단원 3(메모리)과 한 묶음으로, "데이터 이동을 어떻게 **숨기나(overlap)**"가 주제입니다.
스트림·이벤트로 연산과 전송을 겹치고, 이중 버퍼 파이프라인·CUDA Graph까지 다룹니다.

## 학습 목표
- 스트림(순서 실행)과 다른 스트림 간 **오버랩**, 이벤트 동기화를 이해한다.
- **이중 버퍼 청크 파이프라인**으로 전송-연산을 겹치고 속도를 측정한다.
- 이벤트로 구간을 세분 측정하고, 스트림 수를 스케일링한다.
- (심화) **CUDA Graph 캡처**로 런치 오버헤드를 줄이고, 멀티-GPU를 맛본다.

## 목차
1. [스트림이란](#1)
2. [이벤트로 구간 측정](#2)
3. [다중 스트림 + 이벤트 동기화](#3)
4. [비동기 전송 & pinned memory](#4)
5. [이중 버퍼 청크 오버랩 파이프라인](#5)
6. [이벤트 세분 타이밍](#6)
7. [스트림 수 스케일링](#7)
8. [NVTX/프로파일 연계 + Power Iteration](#8)
9. [(심화) CUDA Graph 캡처](#9)
10. [(심화) 멀티-GPU](#10)
11. [체크포인트](#11)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare
print_env()

<a id="1"></a>
## 1. 스트림이란

**스트림(stream)** 은 *순서대로 실행되는 디바이스 작업의 사슬*입니다. 같은 스트림은 순서 보장,
**다른 스트림끼리는 겹쳐(overlap)** 실행될 수 있습니다. 지정 안 하면 기본 스트림을 씁니다(`cp.cuda.get_current_stream()`).

<img src="images/figures/new_stream_concept.png" width="560">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

In [ ]:
A = cp.cuda.Stream(non_blocking=True)
a = cp.random.random(10_000_000, dtype=cp.float32)
with A:
    s = (cp.sin(a)+1).sum()
A.synchronize()
print('A 스트림 결과:', float(s))

<a id="2"></a>
## 2. 이벤트로 구간 측정

이벤트는 스트림 위 시점 표식입니다. 두 이벤트 사이 시간으로 GPU 구간을 정확히 잽니다.

In [ ]:
a = cp.random.random(30_000_000, dtype=cp.float32)
st=cp.cuda.Event(); ed=cp.cuda.Event()
st.record(); y=(cp.sin(a)+1).sum(); ed.record(); ed.synchronize()
print('구간:', cp.cuda.get_elapsed_time(st,ed),'ms')

<a id="3"></a>
## 3. 다중 스트림 + 이벤트 동기화

여러 스트림에 독립 작업을 올리면 겹칠 수 있고, 스트림 간 의존성은 **이벤트**(`record`/`wait_event`)로 표현합니다.

<img src="images/figures/new_streams_events.png" width="620">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

In [ ]:
a1=cp.random.random(20_000_000,dtype=cp.float32); a2=cp.random.random(20_000_000,dtype=cp.float32)
s1=cp.cuda.Stream(non_blocking=True); s2=cp.cuda.Stream(non_blocking=True)
e=cp.cuda.Event()
with s1:
    t=cp.sin(a1); e.record(s1)
with s2:
    s2.wait_event(e)          # s2는 e 이후 진행
    out=(t+cp.cos(a2)).sum()
s2.synchronize()
print('의존성 결합:', float(out))

<a id="4"></a>
## 4. 비동기 전송 & pinned memory

전송은 기본 블로킹입니다. CuPy 13+는 `cp.asarray(x, blocking=False)`/`cp.asnumpy(x, blocking=False)`로 비동기 전송을 합니다.
겹침 효과를 보려면 host 버퍼가 **pinned(page-locked)** 여야 합니다(`cp.cuda.alloc_pinned_memory`).

In [ ]:
n=8_000_000; itemsize=np.dtype(np.float32).itemsize
pinned=cp.cuda.alloc_pinned_memory(n*itemsize)
h=np.frombuffer(pinned,dtype=np.float32,count=n); h[:]=np.random.rand(n).astype(np.float32)
s=cp.cuda.Stream(non_blocking=True)
with s:
    try: d=cp.asarray(h, blocking=False)
    except TypeError: d=cp.asarray(h)
    y=(d*2+1).sum()
s.synchronize(); print('pinned 비동기 전송 결과:', float(y))

<a id="5"></a>
## 5. 이중 버퍼 청크 오버랩 파이프라인
스트림의 **핵심 활용**입니다. 큰 데이터를 청크로 나눠 여러 스트림에 분산하면,
한 청크의 전송과 다른 청크의 연산이 **겹쳐** 전체 시간이 줄어듭니다. 먼저 순차 기준선과 비교합니다.

In [ ]:
N=64_000_000; CH=4_000_000
itemsize=np.dtype(np.float32).itemsize
pin=cp.cuda.alloc_pinned_memory(N*itemsize)
H=np.frombuffer(pin,dtype=np.float32,count=N); H[:]=np.random.rand(N).astype(np.float32)
def work(d): return cp.sqrt(d*d+1.0)

# 기준선(순차): 한 청크씩 전송->연산->회수 (겹침 없음)
def sequential():
    out=np.empty(N,np.float32)
    for s0 in range(0,N,CH):
        d=cp.asarray(H[s0:s0+CH]); r=work(d); out[s0:s0+CH]=cp.asnumpy(r)
    return out
print_bench(bench(sequential, n_repeat=5, name='sequential'))

**연습 — 이중 버퍼 오버랩 직접 구현**: `H`를 `CH` 청크로 나눠 `nstreams`개 스트림에 분산하고,
각 청크에서 `asarray(blocking=False)`→`work`→`asnumpy(blocking=False)`로 처리해 **순차 대비 속도**를 비교하세요.

In [ ]:
def chunked(nstreams=3):
    # TODO: nstreams개 Stream(non_blocking=True) 생성
    #       각 청크를 streams[i%nstreams]에서 asarray(blocking=False)->work->asnumpy(blocking=False)
    #       모든 스트림 synchronize 후 out 반환
    raise NotImplementedError

# print_bench(bench(lambda: chunked(3), n_repeat=5, name='chunked(3)'))
# print('speedup:', round(cpu_ms(bench(sequential))/cpu_ms(bench(lambda: chunked(3))),2))

<details><summary>💡 해답 보기</summary>

```python
def chunked(nstreams=3):
    out=np.empty(N,np.float32)
    streams=[cp.cuda.Stream(non_blocking=True) for _ in range(nstreams)]
    for i,s0 in enumerate(range(0,N,CH)):
        with streams[i%nstreams]:
            try: d=cp.asarray(H[s0:s0+CH], blocking=False)
            except TypeError: d=cp.asarray(H[s0:s0+CH])
            r=work(d)
            try: out[s0:s0+CH]=cp.asnumpy(r, blocking=False)
            except TypeError: out[s0:s0+CH]=cp.asnumpy(r)
    for st in streams: st.synchronize()
    return out

for k in [1,2,4]:
    print_bench(bench(lambda k=k: chunked(k), n_repeat=5, name=f'chunked({k})'))
# 보통 2~3 스트림에서 이득이 포화됩니다.
```
</details>

<a id="6"></a>
## 6. 이벤트 세분 타이밍

전송 구간과 연산 구간을 이벤트로 분리 측정하면 어디가 병목인지 보입니다.

In [ ]:
e0=cp.cuda.Event(); e1=cp.cuda.Event(); e2=cp.cuda.Event()
e0.record()
d=cp.asarray(H[:CH])      # 전송
e1.record()
r=work(d).sum()           # 연산
e2.record(); e2.synchronize()
print(f'transfer {cp.cuda.get_elapsed_time(e0,e1):.3f} ms | compute {cp.cuda.get_elapsed_time(e1,e2):.3f} ms')

<a id="7"></a>
## 7. 스트림 수 스케일링

독립 연산을 1·2·4·8 스트림에 나눠 올리고 시간을 비교합니다. 단일 커널이 GPU를 이미 채우면 이득이 작습니다.

In [ ]:
arrs=[cp.random.random(20_000_000,dtype=cp.float32) for _ in range(8)]
def run_streams(k):
    streams=[cp.cuda.Stream(non_blocking=True) for _ in range(k)]; outs=[]
    for i,a in enumerate(arrs):
        with streams[i%k]: outs.append((cp.sin(a)+1).sum())
    for st in streams: st.synchronize()
    return outs
for k in [1,2,4,8]:
    print_bench(bench(lambda k=k: run_streams(k), n_repeat=10, name=f'{k} stream(s)'))

<a id="8"></a>
## 8. NVTX/프로파일 연계 + Power Iteration

`cupyx.profiler.time_range`로 구간을 NVTX 라벨링하면 Nsight Systems 타임라인에서 오버랩·idle을 볼 수 있습니다(단원 3 도구).
단원 3의 Power Iteration으로 **동기화 빈도**의 영향을 다시 확인합니다(잦은 `float()`=잦은 host 동기화).

In [ ]:
from cupyx.profiler import time_range
M=cp.random.random((1500,1500),dtype=cp.float32); A=(M+M.T)/2
def power_iter_sync(A, iters=300, check_every=1):
    xp=cp.get_array_module(A); x=xp.ones(A.shape[0],dtype=A.dtype)
    for i in range(iters):
        y=A@x; nrm=xp.linalg.norm(y); x=y/nrm
        if i%check_every==0: _=float(nrm)   # host 동기화
    return x
with time_range('pi_check1', color_id=0):
    print_bench(bench(lambda: power_iter_sync(A,check_every=1),  n_repeat=5, name='sync 매 스텝'))
with time_range('pi_check50', color_id=1):
    print_bench(bench(lambda: power_iter_sync(A,check_every=50), n_repeat=5, name='sync 50스텝마다'))

<details><summary>(선택) Nsight Systems 프로파일링 워크플로</summary>

```python
# %%writefile pi.py  로 스크립트 저장 후 터미널에서:
# nsys profile --capture-range=cudaProfilerApi --capture-range-end=stop -o pi python pi.py
# 생성된 pi.nsys-rep 를 Nsight Systems GUI / Perfetto 에서 열어 타임라인 확인
```
`with cupyx.profiler.profile():` 블록 안에서만 캡처되며, `time_range` 라벨이 타임라인에 표시됩니다.
</details>

<a id="9"></a>
## 9. (심화) CUDA Graph 캡처

📖 [`cupy.cuda.Graph`](https://docs.cupy.dev/en/stable/reference/generated/cupy.cuda.Graph.html) — 반복되는 스트림 작업 시퀀스를 **그래프로 캡처**해 한 번에 실행하면 **커널 런치 오버헤드**가 줄어듭니다.
`stream.begin_capture()` … `g = stream.end_capture()` … `g.launch()`. 캡처 중에는 **host 동기 전송 금지**입니다.

In [ ]:
# 작은 커널을 여러 번 실행하는 반복 시퀀스 (런치 오버헤드 지배)
x=cp.random.random(1_000_000,dtype=cp.float32)
def many_small():
    for _ in range(50): cp.add(x,1.0,out=x)

s=cp.cuda.Stream(non_blocking=True)
try:
    with s:
        s.begin_capture()
        for _ in range(50): cp.add(x,1.0,out=x)
        g=s.end_capture()
    print_bench(bench(many_small, n_repeat=50, name='반복 런치'))
    print_bench(bench(lambda: g.launch(), n_repeat=50, name='CUDA Graph'))
except (AttributeError, RuntimeError) as ex:
    print('이 환경에서는 그래프 캡처를 건너뜁니다:', ex)

<a id="10"></a>
## 10. (심화) 멀티-GPU

GPU가 여러 개면 `with cp.cuda.Device(i):` 로 장치를 전환합니다. 연산 입력은 같은 장치에 있어야 합니다.

In [ ]:
ngpu=cp.cuda.runtime.getDeviceCount()
print('GPU 개수:', ngpu)
if ngpu>=2:
    with cp.cuda.Device(0): a0=cp.random.random(1_000_000,dtype=cp.float32); s0=float(a0.sum())
    with cp.cuda.Device(1): a1=cp.random.random(1_000_000,dtype=cp.float32); s1=float(a1.sum())
    print('device0 sum',s0,'| device1 sum',s1)
else:
    print('단일 GPU 환경 — 멀티-GPU 예제는 건너뜁니다.')

<a id="11"></a>
## 11. 체크포인트

- [ ] 스트림(순서)과 다른 스트림 간 오버랩, 이벤트 동기화를 이해했다
- [ ] **이중 버퍼 청크 파이프라인**으로 전송-연산을 겹쳐 속도를 높였다
- [ ] 이벤트로 전송 vs 연산 구간을 분리 측정했다
- [ ] 스트림 수 스케일링과 동기화 빈도의 영향을 확인했다
- [ ] (심화) CUDA Graph 캡처/멀티-GPU를 시도했다

**Day 1 완료!** 다음은 **`day1_capstone`**(통합 실습) → Day 2 **`07_cupy_kernels`**.